# Neo4j Practice — Building a Movie Knowledge Graph

This notebook is the guided walkthrough for the Neo4j practice session. It
follows the same path as [`../README.md`](../README.md) sections 5–8, but runs
the queries from Python so you can keep the results next to the code.

**Before you start:** the graph must be loaded. From the `05_Neo4J` folder on
your host run:

```bash
docker compose up -d
docker exec -i neo4j cypher-shell -u neo4j -p password < sample_data/load.cypher
```

## What you will do

1. Connect to Neo4j and check what was loaded.
2. Match patterns of nodes and relationships.
3. Filter, project, sort and aggregate.
4. Traverse variable-length paths and find shortest paths.
5. Read a query plan and see what an index changes.

Cells marked **✏️ Your turn** are for you to fill in. Every one of them has a
worked solution in the collapsed block underneath.

---
## 1. Connect

The helper `run()` returns results as a pandas DataFrame, which displays as a
table. `run_raw()` is for writes — it returns the counters telling you how many
nodes or relationships changed.

In [ ]:
# Connect to Neo4j with the official Python driver.
#
# Inside Docker the host is the service name `neo4j`. If you run this
# notebook on your own machine instead, set NEO4J_URI=bolt://localhost:7687.
import os
import pandas as pd
from neo4j import GraphDatabase

URI      = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
USER     = os.getenv('NEO4J_USER', 'neo4j')
PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f'Connected to {URI}')


def run(cypher, **params):
    """Run a Cypher query and return the rows as a pandas DataFrame."""
    with driver.session() as session:
        return pd.DataFrame([r.data() for r in session.run(cypher, **params)])


def run_raw(cypher, **params):
    """Run a write query and return its summary counters."""
    with driver.session() as session:
        return session.run(cypher, **params).consume().counters

### Check what is in the graph

If these come back empty, the load step above has not run yet.

In [ ]:
run('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS nodes ORDER BY label')

In [ ]:
run('MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY rel')

You should see **40 Movie, 60 Person, 40 User, 8 Genre** and
**ACTED_IN 198, DIRECTED 44, FOLLOWS 103, IN_GENRE 81, RATED 546**.

The graph you are querying:

```
(:Person)-[:ACTED_IN {roles}]->(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]-------->(:Movie)
(:User)-[:RATED {rating, rated_year}]->(:Movie)
(:User)-[:FOLLOWS {since}]->(:User)
```

---
## 2. Matching patterns

A Cypher query draws the shape you are looking for. `(m:Movie)` is a node with
a label; `-[:ACTED_IN]->` is a typed, directed relationship.

In [ ]:
run('MATCH (m:Movie) RETURN m.title, m.released ORDER BY m.released DESC LIMIT 10')

### Properties live on relationships too

`roles` describes the *connection* between a person and a movie, not either one
on its own. Binding the relationship to a variable (`r`) lets you return it.

In [ ]:
run('''
MATCH (:Person {name: 'Tom Hanks'})-[r:ACTED_IN]->(m:Movie)
RETURN m.title, r.roles, m.released
ORDER BY m.released
''')

### Two hops: co-actors

Note the second leg is undirected (`<-[:ACTED_IN]-`): we walk *into* the movie
and back *out* to the other actors.

In [ ]:
run('''
MATCH (:Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(co:Person)
RETURN DISTINCT co.name AS coActor, m.title AS sharedFilm
ORDER BY coActor
LIMIT 10
''')

### ✏️ Your turn 1

Find the **genres of the movies Keanu Reeves acted in**, with no duplicates.

*Hint:* the path is `Person → Movie → Genre`.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
run('''
MATCH (:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g:Genre)
RETURN DISTINCT g.name AS genre
ORDER BY genre
''')
```
</details>

### ✏️ Your turn 2

Who **directed** *Cloud Atlas*? (There is more than one answer.)

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
run('''
MATCH (p:Person)-[:DIRECTED]->(:Movie {title: 'Cloud Atlas'})
RETURN p.name AS director
ORDER BY director
''')
```
</details>

### Negation and OPTIONAL MATCH

`NOT (pattern)` filters on the *absence* of a relationship. `OPTIONAL MATCH`
behaves like a LEFT JOIN — rows survive with `None` where nothing matched.

In [ ]:
run('MATCH (m:Movie) WHERE NOT (m)<-[:DIRECTED]-() RETURN m.title AS undirected')

In [ ]:
run('''
MATCH (m:Movie)
OPTIONAL MATCH (m)<-[:DIRECTED]-(d:Person)
RETURN m.title, d.name AS director
ORDER BY m.title
LIMIT 10
''')

---
## 3. Aggregating

Cypher groups implicitly: anything in `RETURN` that is not inside an aggregate
function becomes the grouping key. Here `p.name` groups and `count(m)` aggregates.

In [ ]:
run('''
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
RETURN p.name AS actor, count(m) AS films
ORDER BY films DESC, actor
LIMIT 10
''')

`collect()` gathers values into a list — useful for a cast list.

In [ ]:
run('''
MATCH (m:Movie)<-[:ACTED_IN]-(a:Person)
RETURN m.title, collect(a.name) AS cast, count(*) AS actors
ORDER BY actors DESC
LIMIT 5
''')

### Filtering on an aggregate with WITH

`WHERE` cannot filter an aggregate directly. `WITH` ends one stage and begins
the next — Cypher's equivalent of SQL's `HAVING`, but general.

In [ ]:
run('''
MATCH (:User)-[r:RATED]->(m:Movie)
WITH m, avg(r.rating) AS avgRating, count(r) AS votes
WHERE votes >= 10
RETURN m.title, round(avgRating, 2) AS avgRating, votes
ORDER BY avgRating DESC
LIMIT 10
''')

### ✏️ Your turn 3

Count how many **movies belong to each genre**, most common first.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
run('''
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)
RETURN g.name AS genre, count(m) AS movies
ORDER BY movies DESC
''')
```
</details>

### ✏️ Your turn 4

Find every person who **directed more than one** movie.

*Hint:* aggregate with `WITH`, then filter.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
run('''
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
WITH p, count(m) AS films
WHERE films > 1
RETURN p.name AS director, films
ORDER BY films DESC, director
''')
```
</details>

---
## 4. Traversals and paths

This is what a graph database is for. `-[:ACTED_IN*1..4]-` follows between 1 and
4 hops — the depth does **not** have to be known when you write the query, which
is the part that is genuinely awkward in SQL.

⚠️ Always bound the range (`*..4`, never a bare `*`) or a dense graph will
explore an enormous number of paths.

In [ ]:
run('''
MATCH (tom:Person {name: 'Tom Hanks'})-[:ACTED_IN*1..4]-(other:Person)
WHERE other <> tom
RETURN DISTINCT other.name AS reachable
ORDER BY reachable
LIMIT 15
''')

### Shortest path

The list comprehension turns the path into readable names: people have `name`,
movies have `title`, so `coalesce` picks whichever exists.

In [ ]:
run('''
MATCH path = shortestPath(
  (:Person {name: 'Tom Hanks'})-[:ACTED_IN*..6]-(:Person {name: 'Keanu Reeves'})
)
RETURN [n IN nodes(path) | coalesce(n.name, n.title)] AS hops,
       length(path) AS hopCount
''')

### ✏️ Your turn 5

Find the shortest `FOLLOWS` path from **user 1 to user 20**.

*Note:* an empty result is a legitimate answer — it means they are not connected.

In [ ]:
# YOUR CYPHER HERE -- uncomment the lines below and fill in your query:
# run('''
#
# ''')

<details>
<summary>💡 Show solution</summary>

```python
run('''
MATCH path = shortestPath(
  (:User {user_id: 1})-[:FOLLOWS*..6]->(:User {user_id: 20})
)
RETURN [n IN nodes(path) | n.user_id] AS hops, length(path) AS hopCount
''')
```
</details>

---
## 5. Query plans and indexes

`PROFILE` runs the query and reports what it actually did. The number to watch
is **DB Hits** — how many times the engine touched the store.

The helper below pulls the plan out of the result summary.

In [ ]:
def profile(cypher):
    """Run PROFILE and print the operators with their DB hits."""
    with driver.session() as session:
        plan = session.run('PROFILE ' + cypher).consume().profile

    rows = []

    def walk(node, depth=0):
        rows.append({
            'operator': '  ' * depth + node['operatorType'],
            'dbHits': node['dbHits'],
            'rows': node['rows'],
        })
        for child in node.get('children', []):
            walk(child, depth + 1)

    walk(plan)
    return pd.DataFrame(rows)

### Before: no index on `title`

In [ ]:
profile("MATCH (m:Movie {title: 'The Matrix'}) RETURN m.title")

You should see a **NodeByLabelScan** — all 40 movies read, then filtered down to
one. Total **81 DB hits** to return a single row.

### After: add an index

Index creation is asynchronous, so wait for it to come online before profiling.

In [ ]:
run_raw('CREATE INDEX movie_title IF NOT EXISTS FOR (m:Movie) ON (m.title)')

import time
for _ in range(20):
    state = run("SHOW INDEXES YIELD name, state WHERE name = 'movie_title' RETURN state")
    if not state.empty and state.iloc[0]['state'] == 'ONLINE':
        print('index ONLINE')
        break
    time.sleep(1)

In [ ]:
profile("MATCH (m:Movie {title: 'The Matrix'}) RETURN m.title")

Now it is a **NodeIndexSeek** at **2 DB hits** instead of 81.

On 40 movies this is microseconds either way. On 40 million it is the difference
between a scan and an instant lookup. Note what *did not* change: the traversals
were always fast. An index matters for **finding the starting node**, not for
following relationships once you are there — the key performance intuition for
graph databases.

In [ ]:
# The constraints from load.cypher already created indexes -- look:
run('SHOW INDEXES YIELD name, type, labelsOrTypes, properties RETURN name, type, labelsOrTypes, properties')

In [ ]:
# Clean up the index we added
run_raw('DROP INDEX movie_title IF EXISTS')

---
## 6. Wrap-up

Close the driver when you are done.

In [ ]:
driver.close()
print('Connection closed')

### What to take away

1. **Relationships are data** — typed, directed, and able to carry properties
   like `roles`, so the role lives on the connection where it belongs.
2. **You query by drawing the pattern**, and multi-hop questions stay readable
   where the SQL equivalent accumulates joins.
3. **Variable-length traversal is the differentiator** — depth need not be known
   when the query is written.
4. **`WITH` composes queries**, which is how you filter on aggregates.
5. **Indexes find starting nodes; adjacency does the rest.**

Next: [`Homework.ipynb`](Homework.ipynb) — model a relational schema as a graph
and build it yourself.

Shut the environment down from the host when finished:

```bash
docker compose down -v
```